In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import re
import seaborn as sns
sns.set(style='whitegrid')

# Aging acceleration 

In [ ]:
import pandas as pd
import numpy as np

def compute_age_gap_transitions(df, age_columns, age_status_col):
    """
    For each column in `age_columns` (e.g. '40_age_gap', '45_age_gap', ...):
      1) compute μ and σ using only patients with status == 1
      2) define bins [-inf, μ-σ, μ, μ+σ, inf]
      3) cut into 4 groups (0–3), assigning pd.NA to patients with status == 0
      4) build pairwise transition matrices between successive ages,
         using only patients valid at both ages.

    Returns:
      df_grp        – original df + grp_{age} columns (with pd.NA)
      transitions   – dict keyed by (age_i, age_j) of crosstab DataFrames
      thresholds    – dict mapping age (str) → list of bin edges
    """
    df_grp = df.copy()
    labels = [0, 1, 2, 3]
    thresholds = {}
    ages = [c.split('_')[0] for c in age_columns]

    # 1) Compute thresholds & label for each age column
    for i in range(len(age_columns)):
        col = age_columns[i]
        status_col = age_status_col[i]
        age = ages[i]
        grp_col_name = f'grp_{age}'

        # --- MODIFICATION 1: Filter for stats calculation ---
        # Use status col to filter for valid patients to calculate stats
        valid_patients_at_age = df_grp[status_col] == 1
        df_valid = df_grp[valid_patients_at_age]

        # Calculate stats only on valid patients
        μ = df_valid[col].mean()
        σ = df_valid[col].std(ddof=1)

        # Define the 5 bin edges
        bins = [-np.inf, μ - σ, μ, μ + σ, np.inf,]
        thresholds[age] = bins + [len(df_valid)]  # Store thresholds
        # --- MODIFICATION 2: Label only valid patients ---
        # Create the new group column, defaulting to pd.NA
        df_grp[grp_col_name] = pd.NA

        # Apply binning ONLY to the valid patients
        df_grp.loc[valid_patients_at_age, grp_col_name] = pd.cut(
            df_valid[col],  # Use the already-filtered df_valid's column
            bins=bins,
            labels=labels
        )

        # Convert to a nullable integer type to hold 0-3 and pd.NA
        df_grp[grp_col_name] = df_grp[grp_col_name].astype('Int64')

    # 2) Build transition matrices between successive ages
    transitions = {}
    for i in range(len(ages) - 1):
        age_i, age_j = ages[i], ages[i+1]
        g0, g1 = f'grp_{age_i}', f'grp_{age_j}'

        # --- MODIFICATION 3: Filter for valid transitions ---
        # Find patients who are valid (not NA) at BOTH time points
        valid_transition = df_grp[g0].notna() & df_grp[g1].notna()
        df_trans = df_grp[valid_transition]

        # Compute crosstab ONLY on this filtered subset
        mat = pd.crosstab(df_trans[g0], df_trans[g1])

        # Ensure all 4 groups are present in the matrix
        mat = mat.reindex(index=labels, columns=labels, fill_value=0)
        transitions[(age_i, age_j)] = mat

    return df_grp, transitions, thresholds

In [ ]:

ehr_age_trajectory_path = ''
save_path = ''
ehr_age = pd.read_csv(ehr_age_trajectory_path)
ehr_age_followup = pd.read_csv('data/followup_status_by_age.csv')
ehr_age = ehr_age.merge(ehr_age_followup, on='patid', how='left')

# assume ehr_age is your DataFrame with ['40_age_gap', ..., '65_age_gap'] columns
age_cols = ['40_age_gap','45_age_gap','50_age_gap','55_age_gap','60_age_gap','65_age_gap']
status_cols = ['40_followup_status','45_followup_status','50_followup_status','55_followup_status','60_followup_status','65_followup_status']
# 1. compute
df_with_groups, trans, threshold_df = compute_age_gap_transitions(ehr_age, age_cols, status_cols)
# 1) Build the DataFrame
thres_df = pd.DataFrame.from_dict(threshold_df, 
    orient='index',
    columns=['min', 'Mean-std', 'Mean', 'Mean+std', 'max', 'N'])
# 2) Round to 3 decimal places
thres_df = thres_df.round(3)

In [ ]:
# 1) Define the two masks for 40→45 transitions
f1 = '40'
f2 = '45'
mask_1_to_4 = (df_with_groups[f'grp_{f1}'] == 0) & (df_with_groups[f'grp_{f2}'] == 3)
mask_4_to_1 = (df_with_groups[f'grp_{f1}'] == 3) & (df_with_groups[f'grp_{f2}'] == 0)
# 2) Subset the DataFrame
trans_1_to_4 = df_with_groups.loc[mask_1_to_4].copy()
trans_4_to_1 = df_with_groups.loc[mask_4_to_1].copy()
# 3) Inspect counts
print(f"{f1}→{f2}: group 1→4  count = {len(trans_1_to_4)}")
print(f"{f1}→{f2}: group 4→1  count = {len(trans_4_to_1)}")

## disease

In [ ]:


# 2. Common setup: age columns and x‐values
def get_df(disease, ehr_age):
    a = disease # str
    a_40 = ehr_age[ehr_age[a]<=40]
    a_40_other =  ehr_age[~ehr_age.patid.isin(a_40.patid)]
    # a_40_other = a_40_other[a_40_other['40_followup_status'] ==1]

    a_50 = ehr_age[(ehr_age[a]>40) & (ehr_age[a]<=50)]
    a_50_other =  ehr_age[~ehr_age.patid.isin(a_50.patid)]
    a_50_other = a_50_other[~a_50_other.patid.isin(a_40.patid)]
    # a_50_other = a_50_other[a_50_other['50_followup_status'] ==1]

    a_60 = ehr_age[(ehr_age[a]>50) & (ehr_age[a]<=60)]
    a_60_other =  ehr_age[~ehr_age.patid.isin(a_60.patid)]
    a_60_other = a_60_other[~a_60_other.patid.isin(a_50.patid)]
    a_60_other = a_60_other[~a_60_other.patid.isin(a_40.patid)]
    # a_60_other = a_60_other[a_60_other['60_followup_status'] ==1]

    a_70 = ehr_age[(ehr_age[a]>60) & (ehr_age[a]<=70)]
    a_70_other =  ehr_age[~ehr_age.patid.isin(a_70.patid)]
    a_70_other = a_70_other[~a_70_other.patid.isin(a_60.patid)]
    a_70_other = a_70_other[~a_70_other.patid.isin(a_50.patid)]
    a_70_other = a_70_other[~a_70_other.patid.isin(a_40.patid)]
    # a_70_other = a_70_other[a_70_other['65_followup_status'] ==1]
    return disease, a_40, a_40_other, a_50, a_50_other, a_60, a_60_other, a_70, a_70_other


def age_disease_plot(disease, a_40, a_40_other, a_50, a_50_other, a_60, a_60_other, a_70, a_70_other, g=''):
    age_columns = ['40_age_gap','45_age_gap','50_age_gap','55_age_gap','60_age_gap','65_age_gap']
    x = np.array([int(c.split('_')[0]) for c in age_columns])
    # 3. Helper to compute bottom/top quartile groups and their mean+CI
    def get_group_stats(bot_df,top_df):
        n_bot, n_top = len(bot_df), len(top_df)
        def mean_ci(subdf):
            vals = subdf[age_columns].values
            m    = vals.mean(axis=0)
            se   = vals.std(axis=0, ddof=1) / np.sqrt(len(subdf))
            ci   = 1.96 * se
            return m, ci
        bot_mean, bot_ci = mean_ci(bot_df)
        top_mean, top_ci = mean_ci(top_df)
        return (bot_mean, bot_ci, n_bot), (top_mean, top_ci, n_top)
    # 4. Prepare a 1×3 subplot grid (share y-axis)
    fig, axes = plt.subplots(1, 4, figsize=(24, 5), sharey=True)

    # 5. Define the three slices to plot
    panels = [
        ('Diagnosed before 40',    a_40, a_40_other),
        ('Diagnosed between 40-50',    a_50, a_50_other),
        ('Diagnosed between 50-60',    a_60, a_60_other),
        ('Diagnosed between 60-70',    a_70, a_70_other),
        ]

    # 6. Loop and draw each panel
    for ax, (title, subset1, subset2, ) in zip(axes, panels):
        (bot_mean, bot_ci, n_bot), (top_mean, top_ci, n_top) = get_group_stats(subset1, subset2)

        # Bottom quartile: steelblue
        ax.plot(x, top_mean, marker='o', linestyle='-',
                color='steelblue', linewidth=2, label=f'General Population% (n={n_top})')
        ax.fill_between(x, top_mean - top_ci, top_mean + top_ci, color='steelblue', alpha=0.3)

        # Top quartile: indianred
        ax.plot(x, bot_mean, marker='s', linestyle='--',
                color='indianred', linewidth=2, label=f'Disease% (n={n_bot})')
        ax.fill_between(x, bot_mean - bot_ci, bot_mean + bot_ci,
                        color='indianred', alpha=0.3)
        # Reference line & labels
        ax.axhline(0, color='gray', linestyle='--', linewidth=1.5)
        ax.set_title(f'{title}', fontsize=14, fontweight='bold')
        ax.set_xlabel('Chronological Age', fontsize=12)
        if ax is axes[0]:
            ax.set_ylabel('Age Gap', fontsize=12)

        ax.set_xticks(x)
        ax.set_xticklabels(x, fontsize=11)
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        ax.legend(fontsize=11, frameon=True, edgecolor='black', loc='upper left')

    # 7. Final touches
    plt.suptitle(f'{disease[:-4].replace("_", " ")} Age-Gap Trends vs General Population', fontsize=16, fontweight='bold')
    plt.tight_layout(rect=[0,0,1,0.95])
    plt.savefig(save_path + f'/disease_age_trend_{disease}{g}.pdf', dpi=300)
    plt.show()

In [ ]:
disease_age = pd.read_csv('data/cprd_disease_ages.csv') # df, per column is a disease and perrow is a patient age when get the disease, if not na
age_cols = list(disease_age.columns[2:])
a = 'Breast_cancer_age' #  
disease, a_40, a_40_other, a_50, a_50_other, a_60, a_60_other, a_70, a_70_other = get_df(a, ehr_age)
age_disease_plot(disease, a_40, a_40_other, a_50, a_50_other, a_60, a_60_other, a_70, a_70_other)